### Query Enhancement – Query Expansion

Query expansion uses an LLM to rewrite a short or vague user query into a richer one
(synonyms, technical terms, context) before sending it to the retriever.
Better query → better retrieved chunks → better final answer.


In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableMap

In [3]:
loader=TextLoader("langchain_crewai_dataset.txt",encoding="utf-8")
raw_docs=loader.load()
splitter=RecursiveCharacterTextSplitter(chunk_size=300,chunk_overlap=50)
chunks=splitter.split_documents(raw_docs)

In [4]:
print(len(chunks))
chunks[:3]

241


[Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain is an open-source framework designed for developing applications powered by large language models (LLMs). It simplifies the process of building, managing, and scaling complex chains of thought by abstracting prompt management, retrieval, memory, and agent orchestration. Developers can use'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='and agent orchestration. Developers can use LangChain to create end-to-end pipelines that connect LLMs with tools, APIs, vector databases, and other knowledge sources. (v1)'),
 Document(metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='At the heart of LangChain lies the concept of chains, which are sequences of calls to LLMs and other tools. Chains can be simple, such as a single prompt fed to an LLM, or complex, involving multiple conditionally executed steps. LangChain makes it easy to compose and reuse chains using st

In [5]:
embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore=FAISS.from_documents(chunks,embedding_model)
retriever=vectorstore.as_retriever(search_type="mmr",search_kwargs={"k":5})


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001B6B4B9AF90>, search_type='mmr', search_kwargs={'k': 5})

In [7]:
retriever.invoke("LangChain memory")

[Document(id='5db3bc5d-2850-4da2-a5dc-3b5eca349ce3', metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These allow the LLM to maintain awareness of previous conversation turns or summarize long interactions to fit within token limits. (v10)'),
 Document(id='03de69ee-294b-4053-9849-dcd63500af08', metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain workflows are modular and composable. Components like retrievers, memories, agents, and chains can be easily combined and reused. This makes it ideal for building scalable, maintainable LLM applications. (v9)'),
 Document(id='05a63ce5-5ef0-4e82-9165-ab3ea555305d', metadata={'source': 'langchain_crewai_dataset.txt'}, page_content='LangChain offers memory modules like ConversationBufferMemory and ConversationSummaryMemory. These allow the LLM to maintain awareness of previous conversation turns or summarize long 

In [11]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("groq_api")

llm = init_chat_model("groq:openai/gpt-oss-20b")
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001B6D76D7ED0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001B6D7748910>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [12]:
query_expansion_prompt = PromptTemplate.from_template("""
You are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.
Return only the expanded query, no explanation.

Original query: "{query}"

Expanded query:
""")
query_expansion_chain=query_expansion_prompt|llm|StrOutputParser()
query_expansion_chain

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={}, template='\nYou are a helpful assistant. Expand the following query to improve document retrieval by adding relevant synonyms, technical terms, and useful context.\nReturn only the expanded query, no explanation.\n\nOriginal query: "{query}"\n\nExpanded query:\n')
| ChatGroq(metadata={'lc_versions': {'langchain-core': '1.7.0', 'langchain': '1.4.7'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001B6D76D7ED0>, async_client

In [13]:
query_expansion_chain.invoke({"query":"Langchain memory"})

'"LangChain memory" OR "LangChain memory component" OR "LangChain memory module"'

In [14]:
answer_prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")
document_chain=create_stuff_documents_chain(llm=llm,prompt=answer_prompt)

In [15]:
rag_pipeline = (
    RunnableMap({
        "input":   lambda x: x["input"],
        "context": lambda x: retriever.invoke(
            query_expansion_chain.invoke({"query": x["input"]})
        ),
    })
    | document_chain
)

In [16]:
query = {"input": "What types of memory does LangChain support?"}

print("🔍 Expanded query:\n", query_expansion_chain.invoke({"query": query["input"]}))
response = rag_pipeline.invoke(query)
print("\n✅ Answer:\n", response)

🔍 Expanded query:
 LangChain memory types OR memory implementations OR memory backends OR memory modules OR memory adapters OR conversation memory OR retrieval memory OR persistent memory OR in-memory cache OR vector store OR LangChain memory components OR LangChain memory classes OR LangChain memory support OR LangChain memory modules

✅ Answer:
 LangChain supports two primary memory modules that help an LLM keep track of or condense prior dialogue:

| Memory type | What it does |
|-------------|--------------|
| **ConversationBufferMemory** | Stores the entire conversation history (or a configurable window of recent turns) so the model can refer back to previous exchanges. |
| **ConversationSummaryMemory** | Keeps a concise summary of the conversation, allowing the model to remember key points while staying within token limits. |

These modules let developers choose whether to preserve the full dialogue or a summarized version, depending on the use case and token constraints.


In [17]:
query = {"input": "CrewAI agents?"}

print("🔍 Expanded query:\n", query_expansion_chain.invoke({"query": query["input"]}))
response = rag_pipeline.invoke(query)
print("\n✅ Answer:\n", response)

🔍 Expanded query:
 CrewAI agents, CrewAI bots, CrewAI assistants, AI crew management, crew scheduling AI, crew rostering AI, crew resource management AI, airline crew planning AI, maritime crew AI, crew optimization AI, crew workload balancing AI, crew fatigue management AI, crew compliance monitoring AI, crew performance analytics AI, AI-powered crew solutions, CrewAI platform, CrewAI system, CrewAI technology, CrewAI solutions, AI agents, artificial intelligence agents, workflow automation, machine learning, natural language processing, reinforcement learning, crew scheduling optimization, crew rostering optimization, crew management software, CrewAI for airlines, CrewAI for maritime, CrewAI for aviation, CrewAI for crew scheduling, CrewAI for crew planning, CrewAI for crew resource planning, CrewAI for crew compliance, CrewAI for crew safety

✅ Answer:
 **CrewAI agents** are the modular, role‑based AI workers that make up a CrewAI system.  
They are designed to:

1. **Collaborate in